---
# 02. 전처리 — 퍼널 이벤트 정의 & 데이터 가공
---


01_eda에서 본 로그 구조 기준으로 퍼널에 쓸 이벤트(가입완료 / 이력서작성 / 지원완료)를 정의하고 유저 단위 퍼널 테이블을 만든다. 커널 이어서 쓰는 중이면 df 그대로 재사용하면 되고, 새로 켰으면 아래 첫 셀로 다시 로드.

### 2-1. 데이터 재로드 (01_eda와 동일 조건)

In [ ]:
# 01_eda / 02_preprocessing는 같은 커널 세션 안에서 이어 쓰는 걸 전제로 함.
# engine이 이미 정의돼 있지 않다면 (커널을 새로 켠 경우) 01_eda 1-1~1-2 셀을 먼저 실행해서
# import + load_dotenv() + engine 생성을 마친 뒤 이 셀을 실행할 것.
query = """
SELECT user_uuid, URL, timestamp, date, response_code, method
FROM com_2022
WHERE response_code IN ('200', '302')
UNION ALL
SELECT user_uuid, URL, timestamp, date, response_code, method
FROM com_2023
WHERE response_code IN ('200', '302')
"""

df = pd.read_sql(query, engine)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)
df['date'] = pd.to_datetime(df['date'])

# 쿼리 단계에서 이미 response_code 200/302로 필터링되어 있으므로 그대로 사용
df_filtered = df.copy()


### 2-2. Acquisition(가입 완료) 이벤트 정의

- signup/step3/done 외에 complete/github(깃허브 소셜 가입)도 가입완료로 포함시킴

In [ ]:
acquisition = (
    df_filtered[
        df_filtered['URL'].str.split('?').str[0].isin([
            'signup/step3/done',
            'complete/github'
        ])
    ]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'signup_time'})
)
print(f"Acquisition(가입 완료) 유저 수: {len(acquisition):,}명")


### 2-3. Activation ① — 이력서 작성 이벤트 정의 (step1 / step2)

In [ ]:
resume_step1 = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].str.contains(
        'api/users/.+/resume/step1|@.+/resume/step1', regex=True, na=False
    )]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'resume_step1_time'})
)

resume_step2 = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].str.contains(
        'api/users/.+/resume/step2|@.+/resume/step2', regex=True, na=False
    )]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'resume_step2_time'})
)


### 2-4. Activation ② — 지원 퍼널 이벤트 정의 (step1~4 / 완료)

In [ ]:
apply_steps = {}
for step in ['step1', 'step2', 'step3', 'step4']:
    apply_steps[step] = (
        df_filtered[df_filtered['URL'].str.split('?').str[0].isin([
            f'jobs/id/apply/{step}',
            f'api/jobs/id/apply/{step}'
        ])]
        [['user_uuid', 'timestamp']]
        .groupby('user_uuid')['timestamp'].min()
        .reset_index()
        .rename(columns={'timestamp': f'apply_{step}_time'})
    )

apply_complete = (
    df_filtered[df_filtered['URL'].str.split('?').str[0].isin([
        'jobs/id/apply/complete'
    ])]
    [['user_uuid', 'timestamp']]
    .groupby('user_uuid')['timestamp'].min()
    .reset_index()
    .rename(columns={'timestamp': 'apply_complete_time'})
)


### 2-5. 유저 단위 퍼널 테이블 병합 및 전환 순서 검증

In [ ]:
funnel = acquisition.copy()
funnel = funnel.merge(resume_step1, on='user_uuid', how='left')
funnel = funnel.merge(resume_step2, on='user_uuid', how='left')
for step in ['step1', 'step2', 'step3', 'step4']:
    funnel = funnel.merge(apply_steps[step], on='user_uuid', how='left')
funnel = funnel.merge(apply_complete, on='user_uuid', how='left')

# 이전 단계보다 이후 시점에 발생한 경우에만 유효한 전환으로 인정
funnel['did_resume1']  = funnel['resume_step1_time']   >= funnel['signup_time']
funnel['did_resume2']  = funnel['resume_step2_time']   >= funnel['resume_step1_time']
funnel['did_apply1']   = funnel['apply_step1_time']    >= funnel['signup_time']
funnel['did_apply2']   = funnel['apply_step2_time']    >= funnel['apply_step1_time']
funnel['did_apply3']   = funnel['apply_step3_time']    >= funnel['apply_step2_time']
funnel['did_apply4']   = funnel['apply_step4_time']    >= funnel['apply_step3_time']
funnel['did_complete'] = funnel['apply_complete_time'] >= funnel['apply_step1_time']

print(f"퍼널 테이블 shape: {funnel.shape}")
funnel.dtypes


### 2-6. URL → 기능 카테고리 분류 함수

미전환 유저 마지막 행동이나 URL 트래픽을 기능별로 묶어볼 때 쓰는 함수

In [ ]:
def categorize_url(url):
    if url is None:
        return '기타'
    elif 'signup' in url:
        return '가입 프로세스'
    elif 'resume' in url:
        return '이력서 작성'
    elif 'apply' in url:
        return '지원 프로세스'
    elif 'jobs' in url or 'api/jobs' in url:
        return '채용공고 탐색'
    elif 'companies' in url or 'api/companies' in url:
        return '기업 탐색'
    elif 'search' in url:
        return '검색'
    elif 'users' in url or '@user' in url:
        return '프로필/설정'
    elif 'notification' in url:
        return '알림'
    elif 'setting' in url or 'email' in url:
        return '설정'
    else:
        return '기타'
